In [ ]:
from unittest import skip

import pandas as pd
import numpy as np

# Daten einlesen

In [ ]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")
df.head()

## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in df:
    if col.startswith('p-') or col.endswith('-pos_label'):
        clean_name = col.replace('_label', '')
        scores = []
        for i, row in df_with_positions.iterrows():
            position = row[col]
            ja_proz = row['volkja-proz']

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in ["Befürwortend", "Vorzug Volksinitiative"]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in ["Ablehnend", "Vorzug Gegenentwurf"]:
                scores.append((50 - ja_proz) / 100)

            # Neutral
            elif position in ["Keine Empfehlung", "Stimmfreigabe", "Existiert nicht", "Neutral", "Leer einlegen"]:
                scores.append(np.nan)

            # Kategorie zum Fehler abfangen
            else:
                scores.append(np.nan)

        neue_spalten[f"zustimmung_{clean_name}"] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

df_with_positions.head()

In [ ]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)
print(df_with_positions)

### Datenset für Heatmap

In [ ]:
df_pos = df.copy() # Kopie des Datensets
parteien_cols = [c for c in df_pos.columns if str(c).startswith("p-")] # Liste der Partei-Spalten
position_cols = ['br-pos_label','bv-pos_label']+parteien_cols # verschiedene Positionen zusammenhängen.

canton_cols = [c for c in df.columns if str(c).endswith('-japroz')] # Liste der Kanton-Spalten

rows = [] #leere Liste für die Zeilen

# Dreistufige Iteration über die Zeilen, die Positions-Spalten (Parteien, br-pos, bv-pos) und die Kantone
for idx, row in df_pos.iterrows(): # 1. Iteration Zeile (Abstimmung)
    for partei in position_cols: # 2. Iteration Positions-Spalte
        zeile = {
            "abstimmung": idx, #Spalte Abstimmung
            "partei": partei, #Spalte Partei / Institution
        }
        for kanton in canton_cols: # 3. Iteration Kanton
            ja_proz = row[kanton]
            position = row[partei]
            # Ja-Parole oder Volksinitiative bevorzugt
            if position in ["Befürwortend", "Vorzug Volksinitiative"]:
                zeile[kanton] = (ja_proz - 50)
            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in ["Ablehnend", "Vorzug Gegenentwurf"]:
               zeile[kanton] = (50 - ja_proz)
            # Neutral
            elif position in ["Keine Empfehlung", "Stimmfreigabe", "Existiert nicht", "Neutral", "Leer einlegen"]:
                zeile[kanton] = 0.0 # 0 oder nan? @ Manuel  bitte prüfen

            else:
                zeile[kanton] = np.nan
        rows.append(zeile)

df_heatmap = pd.DataFrame(rows)
df_heatmap_with_positions = df_heatmap.drop(columns=["abstimmung"]).groupby("partei").mean(numeric_only=True) # Gruppieren nach Partei und Berechnen des Mittelwerts.
df_heatmap_with_positions.head()

In [ ]:
df_heatmap_with_positions.to_csv('../data/processed/df_heatmap_with_positions.csv', index=True)